# Lab 3: Early Fusion + MLP Classification (MSCOCO)

In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# Load features
image_features = np.load("coco_features/image_features.npy")
caption_features = np.load("coco_features/caption_features.npy")
labels = np.load("coco_features/labels.npy")

# Early fusion
X = np.concatenate([image_features, caption_features], axis=1)  # (5000, 2816)
y = labels  # (5000, 80)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# MLP model
class MLP(nn.Module):
    def __init__(self, input_dim=2816, hidden_dim=512, output_dim=80):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()  # multilabel
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
epochs = 10
batch_size = 64

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train.size(0))
    total_loss = 0
    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X_train[idx].to(device), y_train[idx].to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    preds = model(X_test.to(device))
    test_loss = criterion(preds, y_test.to(device)).item()
print("Final Test Loss:", test_loss)
